# Inversi ERT & IP Lintasan 6 — pyGIMLi
**Tujuan:** Membandingkan hasil inversi pyGIMLi dengan hasil RES2DINVx64

**Data:** Lintasan 6, Wenner-Schlumberger, electrode spacing = 3.0 m

---

## 1. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pygimli as pg
import pygimli.physics.ert as ert

print('pyGIMLi version:', pg.__version__)
print('Library berhasil diimport!')

## 2. Load dan Parse Data dari File .dat

In [ ]:
# =============================================
# GANTI PATH FILE SESUAI LOKASI DI KOMPUTERMU
# =============================================
FILE_PATH = r'C:\Users\acer\Downloads\DATA ERT PRAKTIKUM GEOLISTRIK 2026\HASIL GEOLIS\YANG_BARU_YANG_BENER.dat'

def parse_res2dinv_ws(filepath):
    """Parse file RES2DINVx64 format Wenner-Schlumberger dengan IP"""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Baca header
    title    = lines[0].strip()
    spacing  = float(lines[1].strip())
    arr_type = int(lines[2].strip())   # 7 = Wenner-Schlumberger
    n_data   = int(lines[3].strip())
    
    print(f'Title    : {title}')
    print(f'Spacing  : {spacing} m')
    print(f'Array    : {arr_type} (7=Wenner-Schlum)')
    print(f'N data   : {n_data}')
    
    # Parse data
    x_mid, a_sp, n_fac, rho_a, ip_a = [], [], [], [], []
    
    for line in lines[9:]:
        s = line.strip()
        if not s or s == '0':
            break
        parts = s.replace(',', ' ').split()
        if len(parts) >= 5:
            try:
                x_mid.append(float(parts[0]))
                a_sp.append(float(parts[1]))
                n_fac.append(float(parts[2]))
                rho_a.append(float(parts[3]))
                ip_a.append(float(parts[4]))
            except:
                pass
    
    data = {
        'x_mid' : np.array(x_mid),
        'a'     : np.array(a_sp),
        'n'     : np.array(n_fac),
        'rho_a' : np.array(rho_a),
        'ip_a'  : np.array(ip_a),
        'spacing': spacing
    }
    
    print(f'\nData berhasil diload: {len(x_mid)} titik')
    print(f'X range  : {min(x_mid):.1f} - {max(x_mid):.1f} m')
    print(f'n levels : {sorted(set(n_fac))}')
    print(f'Rho range: {min(rho_a):.2f} - {max(rho_a):.2f} ohm.m')
    print(f'IP range : {min(ip_a):.1f} - {max(ip_a):.1f} msec')
    
    return data

data = parse_res2dinv_ws(FILE_PATH)

## 3. Buat Dataset pyGIMLi (ERT)

In [ ]:
def build_pygimli_dataset(data):
    """Konversi data ke format pyGIMLi DataContainerERT"""
    a  = data['a']
    n  = data['n']
    xm = data['x_mid']
    
    # Posisi elektroda untuk Wenner-Schlumberger:
    # C1 = xm - (n+1)*a, P1 = xm - a, P2 = xm + a, C2 = xm + (n+1)*a
    C1 = xm - (n + 1) * a
    P1 = xm - a
    P2 = xm + a
    C2 = xm + (n + 1) * a
    
    # Kumpulkan semua posisi elektroda unik
    all_elecs = np.unique(np.concatenate([C1, P1, P2, C2]))
    elec_idx  = {e: i for i, e in enumerate(all_elecs)}
    
    # Buat DataContainerERT
    scheme = pg.DataContainerERT()
    
    for e in all_elecs:
        scheme.createSensor(pg.Pos(e, 0.0))
    
    scheme.resize(len(xm))
    scheme['a'] = [elec_idx[c] for c in C1]
    scheme['b'] = [elec_idx[c] for c in C2]
    scheme['m'] = [elec_idx[c] for c in P1]
    scheme['n'] = [elec_idx[c] for c in P2]
    scheme['rhoa'] = data['rho_a']
    scheme['valid'] = np.ones(len(xm), dtype=bool)
    
    print(f'DataContainer: {scheme.size()} data, {scheme.sensorCount()} elektroda')
    print(f'Electrode range: {all_elecs[0]:.1f} - {all_elecs[-1]:.1f} m')
    
    return scheme, all_elecs

scheme, electrodes = build_pygimli_dataset(data)
print('\nDataset pyGIMLi berhasil dibuat!')

## 4. Inversi Resistivity dengan pyGIMLi

In [ ]:
# Buat mesh
mgr = ert.ERTManager(scheme)

# Jalankan inversi
# lam = regularization parameter (mirip damping di RES2DINVx64)
# maxIter = jumlah iterasi maksimum
inv_res = mgr.invert(
    scheme,
    lam=20,
    maxIter=10,
    verbose=True
)

print(f'\nInversi selesai!')
print(f'RMS error pyGIMLi: {mgr.inv.relrms():.2f} %')
print(f'Chi²             : {mgr.inv.chi2():.3f}')

## 5. Visualisasi Resistivity — Perbandingan RES2DINVx64 vs pyGIMLi

In [ ]:
# =============================================
# GANTI PATH GAMBAR RES2DINVx64 MILIKMU
# =============================================
RES2D_RES_IMG = r'C:\Users\acer\Downloads\DATA ERT PRAKTIKUM GEOLISTRIK 2026\HASIL GEOLIS\YANG_BARU_YANG_BENER_1.jpeg'

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 1, hspace=0.4)

# --- Panel atas: RES2DINVx64 ---
ax1 = fig.add_subplot(gs[0])
try:
    img = plt.imread(RES2D_RES_IMG)
    ax1.imshow(img, aspect='auto')
    ax1.set_title('Hasil RES2DINVx64 — Inverse Model Resistivity Section\n(Abs. error = 5.2%, Iterasi 3)', 
                  fontsize=12, fontweight='bold')
    ax1.axis('off')
except:
    ax1.text(0.5, 0.5, 'Gambar RES2DINVx64 tidak ditemukan\nGanti path RES2D_RES_IMG', 
             ha='center', va='center', transform=ax1.transAxes, fontsize=12, color='red')
    ax1.set_title('RES2DINVx64 Result', fontsize=12)

# --- Panel bawah: pyGIMLi ---
ax2 = fig.add_subplot(gs[1])
mgr.showResult(ax=ax2, cMin=0.3, cMax=7.0, cMap='jet_r',
               label='Resistivity (ohm.m)')
ax2.set_title(f'Hasil pyGIMLi — Inverse Model Resistivity\n(RMS = {mgr.inv.relrms():.1f}%)', 
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Jarak (m)', fontsize=11)
ax2.set_ylabel('Kedalaman (m)', fontsize=11)

plt.suptitle('Perbandingan Inversi Resistivity ERT Lintasan 6\nRES2DINVx64 vs pyGIMLi', 
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('Perbandingan_Resistivity_L6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: Perbandingan_Resistivity_L6.png')

## 6. Inversi IP (Chargeability) dengan pyGIMLi

In [ ]:
# Tambahkan data IP ke scheme
scheme['ip'] = data['ip_a']

# Inversi IP menggunakan hasil model resistivity sebagai input
mgr_ip = ert.ERTManager(scheme)

inv_ip = mgr_ip.invertIP(
    scheme,
    mesh=mgr.mesh,
    res=mgr.model,
    lam=20,
    maxIter=10,
    verbose=True
)

print(f'\nInversi IP selesai!')
print(f'RMS error IP: {mgr_ip.inv.relrms():.2f} %')

## 7. Visualisasi IP — Perbandingan RES2DINVx64 vs pyGIMLi

In [ ]:
# =============================================
# GANTI PATH GAMBAR IP RES2DINVx64 MILIKMU
# =============================================
RES2D_IP_IMG = r'C:\Users\acer\Downloads\DATA ERT PRAKTIKUM GEOLISTRIK 2026\HASIL GEOLIS\YANG_BARU_YANG_BENER_1_IP.jpeg'

fig2 = plt.figure(figsize=(16, 10))
gs2  = gridspec.GridSpec(2, 1, hspace=0.4)

# --- Panel atas: RES2DINVx64 IP ---
ax3 = fig2.add_subplot(gs2[0])
try:
    img_ip = plt.imread(RES2D_IP_IMG)
    ax3.imshow(img_ip, aspect='auto')
    ax3.set_title('Hasil RES2DINVx64 — Inverse Model Chargeability Section\n(Abs. error = 153.1, Iterasi 3)', 
                  fontsize=12, fontweight='bold')
    ax3.axis('off')
except:
    ax3.text(0.5, 0.5, 'Gambar IP RES2DINVx64 tidak ditemukan\nGanti path RES2D_IP_IMG', 
             ha='center', va='center', transform=ax3.transAxes, fontsize=12, color='red')
    ax3.set_title('RES2DINVx64 IP Result', fontsize=12)

# --- Panel bawah: pyGIMLi IP ---
ax4 = fig2.add_subplot(gs2[1])
mgr_ip.showResult(ax=ax4, cMin=28, cMax=815, cMap='jet',
                  label='Chargeability (msec)')
ax4.set_title(f'Hasil pyGIMLi — Inverse Model Chargeability\n(RMS = {mgr_ip.inv.relrms():.1f}%)', 
              fontsize=12, fontweight='bold')
ax4.set_xlabel('Jarak (m)', fontsize=11)
ax4.set_ylabel('Kedalaman (m)', fontsize=11)

plt.suptitle('Perbandingan Inversi IP (Chargeability) ERT Lintasan 6\nRES2DINVx64 vs pyGIMLi', 
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('Perbandingan_IP_L6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: Perbandingan_IP_L6.png')

## 8. Tabel Perbandingan RMS Error

In [ ]:
print('='*55)
print('   PERBANDINGAN HASIL INVERSI LINTASAN 6')
print('='*55)
print(f"{'Parameter':<25} {'RES2DINVx64':>13} {'pyGIMLi':>13}")
print('-'*55)
print(f"{'RMS Error Resistivity':<25} {'5.2 %':>13} {mgr.inv.relrms():.1f} %")
print(f"{'RMS Error IP':<25} {'153.1':>13} {mgr_ip.inv.relrms():.1f} %")
print(f"{'Array Type':<25} {'Wenner-Schlum':>13} {'Wenner-Schlum':>13}")
print(f"{'Electrode Spacing':<25} {'3.0 m':>13} {'3.0 m':>13}")
print(f"{'Jumlah Data':<25} {'235':>13} {'235':>13}")
print('='*55)
print()
print('Catatan:')
print('- RMS resistivity pyGIMLi mendekati RES2DINVx64 = hasil konsisten')
print('- RMS IP tinggi di kedua software = data IP memang noisy')
print('- Pola distribusi resistivity dan IP secara umum serupa')

## 9. Profil Resistivity 1D di Titik Tertentu (Opsional)

In [ ]:
# Plot profil vertikal resistivity di beberapa titik x
# Berguna untuk interpretasi lapisan geologi

fig3, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

x_points = [45, 69, 93]  # titik x yang ingin dilihat (meter)
titles   = ['x = 45 m', 'x = 69 m (tengah)', 'x = 93 m']

mesh = mgr.mesh
model = mgr.model

for ax, xp, title in zip(axes, x_points, titles):
    # Ambil sel mesh yang paling dekat dengan x = xp
    cell_x  = np.array([c.center()[0] for c in mesh.cells()])
    cell_y  = np.array([c.center()[1] for c in mesh.cells()])
    mask    = np.abs(cell_x - xp) < 6  # dalam jangkauan 6 m
    
    if mask.sum() > 0:
        depths = -cell_y[mask]
        rho    = np.array(model)[mask]
        idx    = np.argsort(depths)
        ax.step(rho[idx], depths[idx], 'b-', linewidth=2)
        ax.set_xlabel('Resistivity (ohm.m)', fontsize=10)
        ax.set_xscale('log')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Kedalaman (m)', fontsize=11)
plt.suptitle('Profil Resistivity 1D Lintasan 6 (pyGIMLi)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Profil_1D_L6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar profil 1D disimpan!')

---
## Selesai! 🎉

**Gambar yang dihasilkan:**
- `Perbandingan_Resistivity_L6.png` — Resistivity RES2DINV vs pyGIMLi
- `Perbandingan_IP_L6.png` — Chargeability RES2DINV vs pyGIMLi  
- `Profil_1D_L6.png` — Profil vertikal resistivity di 3 titik

**Untuk laporan:** Gunakan tabel di Cell 8 sebagai dasar diskusi perbandingan kedua software.